# 1. 环境配置

## 1.1 python 环境准备

In [1]:
! pip install openai==2.11.0 dashscope==1.25.4 langchain-classic==1.0.0 langchain==1.2.0 langchain-community==0.4.1 langchain-openai==1.1.6 arxiv==2.3.1

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/90/7f/340847023184305a6378d75ec71e1dd38a942dfe71b7c29314b8fbe26948/arxiv-2.3.1-py3-none-any.whl (11 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/4e/eb/c96d64137e29ae17d83ad2552470bafe3a7a915e85434d9942077d7fd011/feedparser-6.0.12-py3-none-any.whl (81 kB)
  Using cached sgmllib3k-1.0.0-py3-none-any.whl

   ---------------------------------------- 3/3 [arxiv]



## 1.2 大模型密钥准备

请根据第一章内容获取相关平台的 API KEY，如若未在系统变量中填入，请将 API_KEY 信息写入以下代码（若已设置请忽略）：

In [2]:
import os

# os.environ["OPENAI_API_KEY"] = "sk-xxxxxxxx"
# os.environ["DASHSCOPE_API_KEY"] = "sk-yyyyyyyy"

## 1.3 实践代码

为了能够顺利的演示内置中间件的使用详情，这里我们使用一段简单的智能体代码演示：

In [3]:
from langchain_community.chat_models import ChatTongyi
import os
llm = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-turbo")

from langchain_community.agent_toolkits.load_tools import load_tools
tools = load_tools(["arxiv"])

from langgraph.checkpoint.memory import InMemorySaver 
memory = InMemorySaver()

from langchain.agents import create_agent
agent = create_agent(model=llm, 
                     tools=tools, 
                     system_prompt="You are a helpful assistant", 
                     checkpointer=memory)

result1 = agent.invoke({"messages": [{"role": "user", "content": "请使用 arxiv 工具查询论文编号 1605.08386"}]}, config={"configurable": {"thread_id": "user_1"}})
print(result1["messages"][-1].content)

论文编号 1605.08386 的信息如下：

- **发表日期**: 2016-05-26
- **标题**: Heat-bath random walks with Markov bases
- **作者**: Caprice Stanley, Tobias Windisch
- **摘要**: 研究了由有限集的允许移动构成的格点图，这些移动可以是任意长度。我们证明了在固定整数矩阵的纤维上，这些图的直径可以被常数从上方界。然后研究了这些图上的热浴随机游走的混合行为。我们还给出了移动集的显式条件，使得热浴随机游走（Glauber动力学的一种推广）在固定维度下是一个扩展器。


# 2. 多智能体应用开发

## 2.1 简介

在前面的学习中，我们主要介绍了：
- LangChain 的内置工具
- 自定义工具的实现方式
- MCP 工具的实现方式

此外，在之前关于内置中间件的介绍中，也有许多与“工具优化”密切相关的内容，例如：
- ToolCallLimitMiddleware：限制工具调用频率
- ToolRetryMiddleware：工具失败时自动重试
- LLMToolEmulator：用LLM模拟工具执行

其中，最后一个 LLMToolEmulator 体现出一个关键思路：大模型本身，就可以是一种能力极强的“工具”。除了输出文本字符串以外，通过结构化输出可辅助我们完成更多复杂任务。

很多人可能会疑惑：既然 agent 的“大脑”本身就是一个大模型，为什么还需要再将另一个大模型作为工具来使用呢？主要有以下几个原因：
- 首先，主智能体模型通常是固定的，难以根据不同任务动态调整。而如果我们将大模型作为一个独立工具，就可以为它自定义专属的系统提示词，甚至使用个性化微调的模型，从而让其更贴合具体任务的需求。
- 其次，所谓的“大模型工具”其实也可以是一个带有工具链的智能体。这样，我们就能将主智能体的一部分功能拆分出去，让各个子模型各司其职，从而形成更高效的分工协作体系。
- 最后，agent 在执行多轮推理时会积累大量上下文信息。这些上下文在某些场景下可能会干扰模型的输出，导致结果不够聚焦。将大模型或智能体作为工具调用，可以有效隔离上下文，使它只聚焦于当前任务，从而提高输出的准确性和可控性。

## 2.2 大模型工具
假如单纯希望 LLM 作为工具，可以使用直接定义函数工具实现：

In [4]:
from langchain_core.tools import tool

@tool
def final_tool(request: str) -> str:
    '''请在所有流程结束后，使用该工具返回最终结果，无论前面工具如何，最后一定要使用该工具'''
    prompt = f"请用100字总结该部分内容，并返回最终结果，内容为：{request}"
    result = llm.invoke(prompt)
    return result.content


tools = [final_tool]

agent = create_agent(model=ChatTongyi(model="qwen-max"), 
           tools=tools, 
           system_prompt="请合理的使用工具完成任务")

result1 = agent.invoke({"messages": [{"role": "user", "content": "请总结一下什么是 AI，最后务必使用工具"}]})
print(result1)
print(result1["messages"][-1].content)

{'messages': [HumanMessage(content='请总结一下什么是 AI，最后务必使用工具', additional_kwargs={}, response_metadata={}, id='fb00a91a-d522-4eed-b162-2cac4d34b069'), AIMessage(content='AI，全称为人工智能（Artificial Intelligence），是指由人制造出来的具有一定智能的系统，能够理解、学习并执行通常需要人类智能才能完成的任务。这些任务包括但不限于语言理解、学习、规划、问题解决、知识表示、感知、模式识别、逻辑推理以及操作物体等。随着技术的发展，AI 已经被广泛应用于多个领域，如语音助手、自动驾驶汽车、医疗诊断、金融分析、客户服务和娱乐等。\n\n为了完成您的请求，我将使用 `final_tool` 来结束这个流程。', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"request": "已经总结了什么是AI，现在使用final_tool来结束流程。"}', 'name': 'final_tool'}, 'id': 'call_f9d5dc70cf1949eea2c72c', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '6b9a93a9-2d75-487a-93b4-38ba92ae3470', 'token_usage': {'input_tokens': 262, 'output_tokens': 143, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 405}}, id='lc_run--019b2fd0-0d16-78f1-9417-c6697843f3be-0', tool_calls=[{'name': 'final_tool', 'args': {'request': '已经总结了什么是AI，现在使用final_tool来结束流程

对于一个智能体而言，假如我们提供给其各种各样的工具，包括处理日历、邮件、搜索、翻译等功能。但是当我们运行的轮数变多时，可能会出现以下问题：
- 认知负担过重
- 上下文信息太杂
- 缺乏专业能力
- 控制流程太复杂

所以其实我们可以对一个“大而全”的智能体进行拆分，用多个小而专的 Agent 协作来完成一个大任务，而不是让一个万能 Agent 全部搞定。

## 2.3 Tool Calling 多智能体系统
假如使用 Tool Calling 的方式构建多智能体系统，通常分为以下几步：
- 创建底层工具
- 构建子智能体（Sub-agent）
- 将子代理包装成工具（给 Supervisor 使用）
- 构建 Supervisor Agent（调度者）
- 调用 Supervisor Agent 

### 2.3.1 创建底层工具

其实前面很多节课程里我们都已经提到了什么是工具。在 Agent 中，Tool 是可以被 agent 调用的“动作”——通常对应一个函数。

比如对于前面提到的任务：“请帮我安排明天的行程，并发邮件告诉我朋友。”

我们可以将这个任务的实现拆分为以下几个工具：
- 查询空闲时间 get_available_time_slots
- 创建日程 create_calendar_event
- 发送邮件 send_email

当然由于这里只是演示，后续的工具都只是模拟真实场景而已，但是大家可以在接下来的工具中添加真实的逻辑。

#### 查询空闲时间 get_available_time_slots ：

In [5]:
@tool
def get_available_time_slots(
  attendees: list[str],
  date: str, # ISO format: "2024-01-15"
  duration_minutes: int
) -> list[str]:
  """Check calendar availability for given attendees on a specific date."""
  return ["09:00", "14:00", "16:00"]

#### 创建日程 create_calendar_event ：

In [6]:
@tool
def create_calendar_event(
  title: str,
  start_time: str, # ISO format: "2024-01-15T14:00:00"
  end_time: str,  # ISO format: "2024-01-15T15:00:00"
  attendees: list[str], # email addresses
  location: str = ""
) -> str:
  """Create a calendar event. Requires exact ISO datetime format."""
  return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"

#### 发送邮件 send_email ：

In [7]:
@tool
def send_email(
  to: list[str],   # email addresses
  subject: str,
  body: str,
  cc: list[str] = []
) -> str:
  """Send an email via email API. Requires properly formatted addresses."""
  return f"Email sent to {', '.join(to)} - Subject: {subject}"

### 2.3.2 构建子智能体（sub-agents）
所谓的子智能体其实是一个只负责「一个垂直任务」的智能体，它自己可以使用工具，但不与用户直接对话。

那我们这里可以根据任务需要将其拆分为两个子智能体：
- 日程管理子智能体 calendar_agent：基于创建日程 create_calendar_event 和查询空闲时间 get_available_time_slots 两个工具实现精准的日常管理工作。
- 邮件发送子智能体 email_agent：基于发送邮件 send_email 工具实现对于指定用户发送特定内容的信息的工作。

和之前构建智能体一样，我们需要先去设置一下其所使用的大模型、工具以及系统提示词（子智能体由于只需要完成指定任务不需要设置记忆）。

#### 构建日程管理子智能体 calendar_agent：

In [8]:
from langchain_community.chat_models import ChatTongyi
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=(
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
    )
)

然后我们可以测试一下其具体的效果（这里使用的是流式输出的方式）：

In [9]:
query = "Schedule a team meeting next Tuesday at 2pm for 1 hour"

for step in calendar_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  create_calendar_event (call_f53399f3f9f344ea9c1fb1)
 Call ID: call_f53399f3f9f344ea9c1fb1
  Args:
    title: Team Meeting
    start_time: 2023-11-21T14:00:00
    end_time: 2023-11-21T15:00:00
    attendees: ['team@company.com']
    location:
================================= Tool Message =================================
Name: create_calendar_event

Event created: Team Meeting from 2023-11-21T14:00:00 to 2023-11-21T15:00:00 with 1 attendees
================================== Ai Message ==================================

The team meeting has been scheduled for next Tuesday, November 21, 2023, from 2:00 PM to 3:00 PM.


#### 构建发送邮件子智能体 email_agent：


In [10]:
from langchain_community.chat_models import ChatTongyi
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=(
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
    )
)

然后我们也可以尝试着对其进行调用：

In [11]:
query = "Send the design team a reminder about reviewing the new mockups"

for step in email_agent.stream({"messages": [{"role": "user", "content": query}]}):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  send_email (call_5688cc3ed300484f9175a5)
 Call ID: call_5688cc3ed300484f9175a5
  Args:
    to: ['design.team@example.com']
    subject: Reminder: Review New Mockups
    body: Hi Design Team, 

Just a friendly reminder to review the new mockups that were shared with you earlier this week. Your feedback is important for us to proceed to the next stage of the project. 

Please share your thoughts by the end of the day tomorrow. 

Best regards,
[Your Name]
================================= Tool Message =================================
Name: send_email

Email sent to design.team@example.com - Subject: Reminder: Review New Mockups
================================== Ai Message ==================================

The email has been sent to the design team at design.team@example.com with the subject "Reminder: Review New Mockups". The body of the email kindly reminds them to review the new mockups an

### 2.3.3 将子代理包装成工具（给 Supervisor 调用）
虽然 calendar_agent 和 email_agent 已经可以独立工作，但 Supervisor_agent 是通过「调用工具」的方式来使用它们的。所以我们还要把子代理包装成新的高层工具，供上层 Supervisor_agent 调度。

所以我们刚刚的两个 agent 会被包装成两个工具：
- schedule_event
- manage_email

当然我们也需要写上对应的函数文档字符串来告诉 Supervisor 在什么时候去使用该工具。

#### 构建子代理工具 schedule_event：

In [12]:
@tool
def schedule_event(request: str) -> str:
  """Schedule calendar events using natural language.

  Use this when the user wants to create, modify, or check calendar appointments.
  Handles date/time parsing, availability checking, and event creation.

  Input: Natural language scheduling request (e.g., 'meeting with design team
  next Tuesday at 2pm')
  """
  result = calendar_agent.invoke({
    "messages": [{"role": "user", "content": request}]
  })
  return result["messages"][-1].text

#### 构建子代理工具 manage_email：

In [13]:
@tool
def manage_email(request: str) -> str:
  """Send emails using natural language.

  Use this when the user wants to send notifications, reminders, or any email
  communication. Handles recipient extraction, subject generation, and email
  composition.

  Input: Natural language email request (e.g., 'send them a reminder about
  the meeting')
  """
  result = email_agent.invoke({
    "messages": [{"role": "user", "content": request}]
  })
  return result["messages"][-1].text

### 2.3.4 构建 Supervisor Agent（调度者）

在构建好了 Supervisor 可使用的工具后，我们就可以创建 Supervisor_agent 并准备后续进行调用了，其具体的职责并不是直接使用底层工具，而是：
- 理解用户自然语言请求
- 拆解任务
- 调用合适的子代理工具
- 整合并回复结果

所以根据这个需求，我们可以按以下方式创建 Supervisor_agent :

In [14]:
from langchain_community.chat_models import ChatTongyi
model = ChatTongyi(api_key=os.environ.get("DASHSCOPE_API_KEY"), model="qwen-max")

supervisor_agent = create_agent(
  model,
  tools=[schedule_event, manage_email],
  system_prompt=(
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence."
  )
)

### 2.3.5 调用 Supervisor Agent

假如我们就希望其进行会议的安排：

In [15]:
query = "Schedule a team standup for tomorrow at 9am"

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_175541f8bde048569f824d)
 Call ID: call_175541f8bde048569f824d
  Args:
    request: Schedule a team standup for tomorrow at 9am
================================= Tool Message =================================
Name: schedule_event

The team standup has been scheduled for tomorrow at 9 AM and will last until 9:30 AM. All team members have been invited to the event.
================================== Ai Message ==================================

The team standup has been scheduled for tomorrow at 9 AM and will last until 9:30 AM. All team members have been invited to the event.


那这个时候就会单独只调用其中一个工具（schedule_event）来进行完成。

假如我们希望其能够其能够同时安排会议并发邮件提醒，我们也可以进行实现：

In [16]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

for step in supervisor_agent.stream(
    {"messages": [{"role": "user", "content": query}]}
):
    for update in step.values():
        for message in update.get("messages", []):
            message.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  schedule_event (call_8055ee9de1824ef29141c2)
 Call ID: call_8055ee9de1824ef29141c2
  Args:
    request: Schedule a meeting with the design team next Tuesday at 2pm for 1 hour
  manage_email (call_9c2d20d4d4584c09a5e992)
 Call ID: call_9c2d20d4d4584c09a5e992
  Args:
    request: send an email reminder to the design team about reviewing the new mockups before the meeting
================================= Tool Message =================================
Name: schedule_event

The meeting with the design team has been scheduled for next Tuesday, October 10th, from 2 PM to 3 PM.
================================= Tool Message =================================
Name: manage_email

The email reminder has been sent to the design team at `design.team@example.com` with the subject "Reminder: Review New Mockups Before the Meeting". The body of the email kindly requests them to review the new mockups before t

可以看到返回的内容为：
- Supervisor_agent 分析了任务后一次性发出了两条工具调用的指令
- 然后子代理 schedule_event 和 manage_email 分别返回了工具调用的结果（非直接工具返回的结果）
- 最后 Supervisor_agent 根据工具调用的信息对问题进行了回复

## 2.4 总结

整个 Tool Calling 模式的 Multi-Agent System 由三层组成：
- 底层：是一些严格格式的 API 工具，它们要求精确的输入格式；
- 中间层：是子代理（Sub-agent），可以接收自然语言请求，将其转换为结构化 API 调用，并返回自然语言的确认回复；
- 顶层：是 Supervisor（监督者代理），负责调用这些高层功能（子代理工具）并整合最终结果。

这种“职责分离”的架构带来了多个优势：
- 每一层都有清晰、专注的职责；
- 可以在不影响已有模块的前提下，轻松添加新领域；
- 各层之间解耦，便于独立测试与持续优化。

## 2.5 多智能体优化

假如我们希望对当前 Tool Calling 模式的 Multi-Agent System 进行优化，我们可以从三个方面入手：
- 增加子智能体能够获取到的信息
- 增加主智能体能够从子智能体中获取到的信息（也就是子智能体返回更多信息）
- 对于一些危险的工具添加人工审核以避免风险
|
### 2.5.1 增加子智能体获取的信息

在当前的代码里， sub_agent 能看到的信息只是 Supervisor_agent 发送的查询请求。但假如我们希望让子代理看到“上层全部对话历史”或其它状态信息，我们可以通过工具调用时的 ToolRuntime 注入状态：

In [17]:
from langchain.tools import tool, ToolRuntime

@tool
def schedule_event(request: str, runtime: ToolRuntime) -> str:
  """Schedule calendar events using natural language.

  Use this when the user wants to create, modify, or check calendar appointments.
  Handles date/time parsing, availability checking, and event creation.

  Input: Natural language scheduling request (e.g., 'meeting with design team
  next Tuesday at 2pm')
  """
  # Customize context received by sub-agent
  original_user_message = next(message for message in runtime.state["messages"] if message.type == "human")
  prompt = ("You are assisting with the following user inquiry:\n\n"
      f"{original_user_message.text}\n\n"
      "You are tasked with the following sub-request:\n\n"
      f"{request}")
  result = calendar_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
  return result["messages"][-1].text

所以这个时候 sub_agent 不再只能接收到调用（request）的信息，还能知道部分当前任务的信息并智能化的做出调用的决定。

### 2.5.2 增加主智能体获取的信息

除了向子代理里注入更多的信息，类似的我们可以让子代理返回更多的信息，比如在 return 的时候并不是仅仅把 result["messages"][-1].text 而是返回一些状态信息：

In [18]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})

    # Option 1: Return just the confirmation message
    return result["messages"][-1].text

    # Option 2: Return structured data
    # return json.dumps({
    #     "status": "success",
    #     "event_id": "evt_123",
    #     "summary": result["messages"][-1].text
    # })

通过 json.dumps ，我们可以把特定格式的 json 内容转为字符串的形式然后再返回给大模型。比如上面的内容会得到： ``` "{\"status\": \"success\", \"event_id\": \"evt_123\", \"summary\": \"The event has been scheduled for next Tuesday at 2 PM.\"}" ```。这样Supervisor_agent 就会知道成功运行且运行的 id 是 evt_123 了。

### 2.5.3 添加人工审核
对于人工审核，可以使用上一章节提到的内置中间件 HumanInTheLoopMiddleware 来实现。这个中间件允许我们能够在“子代理调用工具之前”插入人工判断点。

比如当智能体执行关键操作（如发邮件、创建日程）时，先中断流程，由人类手动“审批 / 编辑 / 拒绝”，再继续执行。

那在我们的系统中，我们也可以添加进该机制，从而实现：
- 确保发出去的邮件不会出错
- 日程安排前想确认时间和人选
- 涉及敏感操作（下单、删库、通知大群）能够提前预判

我们可以给 calendar_agent 添加上人工审核中间件。仅针对 create_calendar_event 工具，而 get_available_time_slots 工具可以自由调用。

In [19]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

calendar_agent = create_agent(
  model,
  tools=[create_calendar_event, get_available_time_slots],
  system_prompt=("You are a calendar scheduling assistant..."),
  middleware=[ 
    HumanInTheLoopMiddleware( 
      interrupt_on={"create_calendar_event": True}, 
      description_prefix="Calendar event pending approval", 
    ), 
  ], 
)

而对于第二个 email_agent 而言，也可以对 send_email 添加上人工审核。

In [20]:
email_agent = create_agent(
  model=llm,
  tools=[send_email],
  system_prompt=("You are an email assistant... "),
  middleware=[ 
    HumanInTheLoopMiddleware( 
      interrupt_on={"send_email": True}, 
      description_prefix="Outbound email pending approval", 
    ), 
  ], 
)

但是在 HumanInTheLoopMiddleware 中有一个要求，就是必须要在有设置记忆的情况下才能够实现人工审核的操作。所以我们可以为 supervisor_agent 添加 checkpointer 参数：

In [21]:
from langgraph.checkpoint.memory import InMemorySaver

supervisor_agent = create_agent(
    model=model,
    tools=[schedule_event, manage_email],
    system_prompt=("You are a helpful personal assistant. "),
    checkpointer=InMemorySaver()  # 🔥 激活状态保存
)

这样子，在运行过程中的信息就能够保存下来了，那么在传入人工的指令后也可以基于之前的记忆继续恢复了。

对于子代理而言，由于其本身就是为了完成任务而设置的，除非是一些需要基于上下文完成任务的智能体工具，不然是不需要添加记忆进去的。添加了反而可能导致状态不一致或冗余。

当然和前面一样，我们一样可以提出问题，然后调用 supervisor_agent 进行回复：

In [22]:
from langchain_core.messages import HumanMessage

query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)

config = {"configurable": {"thread_id": "1"}}

response = supervisor_agent.invoke(
    {"messages": [HumanMessage(content=query)]},
    config=config
)

print(response)

{'messages': [HumanMessage(content='Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.', additional_kwargs={}, response_metadata={}, id='0a420491-6cc7-48bf-9a1e-bc22e2237b69'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"request": "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour"}', 'name': 'schedule_event'}, 'id': 'call_1938984e56b6448bab4d87', 'index': 0, 'type': 'function'}, {'function': {'arguments': '{"request": "send an email to the design team reminding them to review the new mockups before the meeting"}', 'name': 'manage_email'}, 'id': 'call_d1a7a56cf36a4d43b4e3c6', 'index': 1, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '5f6703ed-658e-4161-9089-fffde7d33c7d', 'token_usage': {'input_tokens': 431, 'output_tokens': 67, 'prompt_tokens_details': {'cached_tokens': 0}

In [23]:
from langgraph.types import Command
# 🔁 循环处理多个中断（逐个审批直到流程结束）
while "__interrupt__" in response:
  print("\n🟠 检测到中断，进入人工审核环节...")

  # 🧑 打印用户原始提问
  user_msg = next((m.content for m in response["messages"] if m.type == "human"), "无")
  print(f"\n🧑 用户提问：{user_msg}")

  # 遍历中断对象（可能包含多个 action request）
  for interrupt in response["__interrupt__"]:
    for i, req in enumerate(interrupt.value["action_requests"]):
      print(f"📝 描述：{req.get('description', '无')}")
      print(f"✅ 可选操作：{interrupt.value['review_configs'][i]['allowed_decisions']}")

    # 👤 人工输入决策
    decision_type = input("\n请输入你的决定（approve / edit / reject）：").strip().lower()
    if decision_type not in ["approve", "edit", "reject"]:
      print("⚠️ 无效输入，默认设置为 reject")
      decision_type = "reject"

    decision_payload = {"type": decision_type}

    # ✍️ 如果选择 edit：展示原请求并让用户直接输入完整 JSON
    if decision_type == "edit":
      print("\n🛠️ 当前模型原始调用指令如下：")
      print(json.dumps(req, indent=2, ensure_ascii=False))
      print("\n✍️ 请粘贴你想执行的完整新调用 JSON（例如：）")
      print('{"name": "send_email", "args": {"to": ["x@example.com"], "subject": "测试", "body": "内容"}}')
      try:
        new_raw = input("\n请输入新的调用 JSON：").strip()
        decision_payload["edited_action"] = json.loads(new_raw)
      except Exception as e:
        print(f"❌ JSON 格式错误：{e}")
        print("⚠️ 自动拒绝此调用")
        decision_payload = {"type": "reject", "message": "编辑失败，输入格式错误"}
    # ❌ 如果拒绝，填写理由
    elif decision_type == "reject":
      reason = input("请输入拒绝理由：")
      decision_payload["message"] = reason
    # ✅ 恢复执行（Resume）
    try:
      response = supervisor_agent.invoke(
        Command(resume={interrupt.id: {"decisions": [decision_payload]}}), config=config)
    except Exception as e:
      print(f"❌ 执行恢复时发生错误：{e}")
      break
# ✅ 最终结果输出
print("\n🟢 流程结束，模型最终回复：")
print(response["messages"][-1].content)


🟠 检测到中断，进入人工审核环节...

🧑 用户提问：Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, and send them an email reminder about reviewing the new mockups.
📝 描述：Outbound email pending approval

Tool: send_email
Args: {'body': 'Please review the new mockups before the meeting.', 'subject': 'Reminder to Review New Mockups', 'to': ['design-team@example.com']}
✅ 可选操作：['approve', 'edit', 'reject']
📝 描述：Calendar event pending approval

Tool: create_calendar_event
Args: {'title': 'Design Team Meeting', 'start_time': '2023-04-18T14:00:00', 'end_time': '2023-04-18T15:00:00', 'attendees': ['designer1@example.com', 'designer2@example.com', 'designer3@example.com'], 'location': 'Conference Room B'}
✅ 可选操作：['approve', 'edit', 'reject']

🟢 流程结束，模型最终回复：
The meeting with the design team has been scheduled for next Tuesday, April 18th, from 2:00 PM to 3:00 PM in Conference Room B. All team members have been invited and an email reminder has been sent to them to review the new mockups before t

# 3. Handsoff

In [ ]:
from langchain_community.chat_models import ChatTongyi
from langchain.messages import HumanMessage

model = ChatTongyi(
    model="qwen-max",
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    temperature=0.2,
)

resp = model.invoke([
    HumanMessage(content="你好，用一句话介绍你自己")
])

print(resp)


content='你好，我是Qwen，由阿里云开发的超大规模语言模型，致力于帮助用户获得准确、有用的信息。' additional_kwargs={} response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '1bc68bc9-5552-4b8c-b0ea-103c18ce4e1c', 'token_usage': {'input_tokens': 14, 'output_tokens': 25, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 39}} id='lc_run--019b546e-ec12-77a1-aac0-095222d14445-0'


In [2]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[],
)


result = agent.invoke({
    "messages": [
        HumanMessage(content="你好，请用一句话介绍 LangChain")
    ]
})

print(result)

{'messages': [HumanMessage(content='你好，请用一句话介绍 LangChain', additional_kwargs={}, response_metadata={}, id='5f43bb57-3c5b-45f0-96d8-a8f707e4b195'), AIMessage(content='LangChain 是一个用于开发由语言模型驱动的应用程序的框架，它帮助开发者更高效地构建可定制的、模块化的应用。', additional_kwargs={}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'stop', 'request_id': '661dfe05-5a29-419c-95ae-234b8efff729', 'token_usage': {'input_tokens': 15, 'output_tokens': 30, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 45}}, id='lc_run--019b5471-6185-7830-a32e-9f044cf57a53-0')]}


In [3]:
from langchain.tools import tool

@tool
def say_hello(name: str) -> str:
    """向指定的人打招呼"""
    return f"你好，{name}！很高兴认识你。"

agent = create_agent(
    model=model,
    tools=[say_hello],  # 👈 第一次加 tool
)

result = agent.invoke({
    "messages": [
        HumanMessage(content="请你调用工具，向张三打个招呼")
    ]
})

print(result)

{'messages': [HumanMessage(content='请你调用工具，向张三打个招呼', additional_kwargs={}, response_metadata={}, id='0dbe7cd3-00d2-4233-af15-7129bed38d64'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"name": "张三"}', 'name': 'say_hello'}, 'id': 'call_793e11e9a480410ebe6cf1', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '133cf8d6-10a1-4176-8e29-ecbf062a8643', 'token_usage': {'input_tokens': 238, 'output_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 256}}, id='lc_run--019b5485-cbc8-7732-bd95-3d4471edbdb6-0', tool_calls=[{'name': 'say_hello', 'args': {'name': '张三'}, 'id': 'call_793e11e9a480410ebe6cf1', 'type': 'tool_call'}]), ToolMessage(content='你好，张三！很高兴认识你。', name='say_hello', id='2979b86f-23e7-4d6f-b087-ba43892d1676', tool_call_id='call_793e11e9a480410ebe6cf1'), AIMessage(content='工具已经向张三打了招呼，内容是：“你好，张三！很高兴认识你。”', additional_kwargs={}, response_metadat

In [1]:
from langchain.agents import AgentState
from typing_extensions import NotRequired

class SimpleState(AgentState):
    name: NotRequired[str]

In [2]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def record_name(
    name: str,
    runtime: ToolRuntime[None, SimpleState],
) -> Command:
    """记录用户的名字"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"已记录你的名字：{name}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "name": name,   # 👈 写入 State
        }
    )

In [3]:
from langchain_community.chat_models import ChatTongyi
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatTongyi(
    model="qwen-max",
    temperature=0.2,
)

agent = create_agent(
    model=model,
    tools=[record_name],
    state_schema=SimpleState,      # 👈 告诉 agent：我有 state
    checkpointer=InMemorySaver(),  # 👈 让 state 跨轮保存
)




In [4]:
from langchain.messages import HumanMessage
import uuid

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

# 第一轮：告诉名字
agent.invoke(
    {"messages": [HumanMessage(content="我叫李剑锋")]},
    config
)

# 第二轮：不再重复说名字
result = agent.invoke(
    {"messages": [HumanMessage(content="你记得我叫什么吗？")]},
    config
)

print(result)

{'messages': [HumanMessage(content='我叫李剑锋', additional_kwargs={}, response_metadata={}, id='87b7bfbf-c164-4658-ae8b-970953dd5d53'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"name": "李剑锋"}', 'name': 'record_name'}, 'id': 'call_22e5bdb2ee8b4a13b4cb9e', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen-max', 'finish_reason': 'tool_calls', 'request_id': '7ccab9f2-d43e-4452-9899-a0cf6ccad4a7', 'token_usage': {'input_tokens': 232, 'output_tokens': 19, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 251}}, id='lc_run--019b5abd-1547-7b30-9c26-2774e9cae815-0', tool_calls=[{'name': 'record_name', 'args': {'name': '李剑锋'}, 'id': 'call_22e5bdb2ee8b4a13b4cb9e', 'type': 'tool_call'}]), ToolMessage(content='已记录你的名字：李剑锋', name='record_name', id='3e055c53-9677-4b3b-871e-75948c6282ca', tool_call_id='call_22e5bdb2ee8b4a13b4cb9e'), AIMessage(content='已经记住了你的名字，李剑锋。有什么我可以帮到你的吗？', additional_kwargs={}, response_metadata={'mode

In [5]:
from langchain.agents import AgentState
from typing_extensions import NotRequired
from typing import Literal

Step = Literal["step_a", "step_b"]

class StepState(AgentState):
    current_step: NotRequired[Step]

In [6]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import ToolMessage
from langgraph.types import Command

@tool
def go_to_step_b(
    runtime: ToolRuntime[None, StepState],
) -> Command:
    """切换到 step_b"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content="已进入第二阶段",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "current_step": "step_b",  # 👈 handoff 发生点
        }
    )

In [7]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def step_middleware(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:

    step = request.state.get("current_step", "step_a")

    if step == "step_a":
        system_prompt = "你现在处于第一阶段，只能简单回应，并提示可以进入第二阶段。"
        tools = [go_to_step_b]
    else:
        system_prompt = "你现在处于第二阶段，可以正常自由回答问题。"
        tools = []

    request = request.override(
        system_prompt=system_prompt,
        tools=tools,
    )
    return handler(request)

In [8]:
from langchain_community.chat_models import ChatTongyi
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

model = ChatTongyi(
    model="qwen-max",
    temperature=0.2,
)

agent = create_agent(
    model=model,
    tools=[go_to_step_b],
    state_schema=StepState,
    middleware=[step_middleware],
    checkpointer=InMemorySaver(),
)


In [9]:
from langchain.messages import HumanMessage

result1 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="你好")
        ]
    },
    config
)

print("=== 第一轮 messages ===")
for msg in result1["messages"]:
    msg.pretty_print()


=== 第一轮 messages ===
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么可以帮助你的吗？如果你需要更详细的信息，我们可以进入第二阶段。


In [10]:
result2 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="继续")
        ]
    },
    config
)

print("=== 第二轮 messages ===")
for msg in result2["messages"]:
    msg.pretty_print()


=== 第二轮 messages ===
================================ Human Message =================================

你好
================================== Ai Message ==================================

你好！有什么可以帮助你的吗？如果你需要更详细的信息，我们可以进入第二阶段。
================================ Human Message =================================

继续
================================== Ai Message ==================================
Tool Calls:
  go_to_step_b (call_62670daeab774dfe9c1d65)
 Call ID: call_62670daeab774dfe9c1d65
  Args:
================================= Tool Message =================================
Name: go_to_step_b

已进入第二阶段
================================== Ai Message ==================================

好的，我们现在处于第二阶段。请告诉我你需要什么帮助或有什么问题需要解答？


In [14]:
import os
import uuid
from typing import Callable, Literal
from typing_extensions import NotRequired

from langchain_community.chat_models import ChatTongyi
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool, ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


# =====================
# 0) 模型初始化（ChatTongyi）
# =====================
model = ChatTongyi(
    api_key=os.environ.get("DASHSCOPE_API_KEY"),
    model="qwen-max",
    temperature=0.3,
)


# =====================
# 1) 自定义 State
# =====================
SupportStep = Literal["warranty_collector", "issue_classifier", "resolution_specialist"]


class SupportState(AgentState):
    """客服流程的 state。"""

    current_step: NotRequired[SupportStep]
    warranty_status: NotRequired[Literal["in_warranty", "out_of_warranty"]]
    issue_type: NotRequired[Literal["hardware", "software"]]


# =====================
# 2) 工具：更新 state + 推进阶段
# =====================
@tool
def record_warranty_status(
    status: Literal["in_warranty", "out_of_warranty"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """记录保修状态，并推进到问题分类阶段。"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"保修状态已记录：{status}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "warranty_status": status,
            "current_step": "issue_classifier",
        }
    )


@tool
def record_issue_type(
    issue_type: Literal["hardware", "software"],
    runtime: ToolRuntime[None, SupportState],
) -> Command:
    """记录问题类型，并推进到解决方案阶段。"""
    return Command(
        update={
            "messages": [
                ToolMessage(
                    content=f"问题类型已记录：{issue_type}",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
            "issue_type": issue_type,
            "current_step": "resolution_specialist",
        }
    )


@tool
def escalate_to_human(reason: str) -> str:
    """升级到人工支持。"""
    return f"已为你升级到人工支持。原因：{reason}"


@tool
def provide_solution(solution: str) -> str:
    """提供解决方案。"""
    return f"已提供解决方案：{solution}"


# 可选：回退
@tool
def go_back_to_warranty() -> Command:
    """回到保修确认阶段。"""
    return Command(update={"current_step": "warranty_collector"})


@tool
def go_back_to_classification() -> Command:
    """回到问题分类阶段。"""
    return Command(update={"current_step": "issue_classifier"})


# =====================
# 3) 每个阶段的 prompt + tools 配置
# =====================
WARRANTY_COLLECTOR_PROMPT = """你是一名售后客服助手，正在帮助用户解决设备问题。

【当前阶段：确认保修】
你需要：
1. 友好地问候用户
2. 询问设备是否仍在保修期（或询问购买时间/订单信息用于判断）
3. 一旦信息足够明确，必须调用 record_warranty_status 记录结果并进入下一阶段

要求：语气自然、友好，不要一次问太多问题。"""


ISSUE_CLASSIFIER_PROMPT = """你是一名售后客服助手，正在帮助用户解决设备问题。

【当前阶段：问题分类】
已知信息：保修状态 = {warranty_status}

你需要：
1. 引导用户描述问题现象
2. 判断问题属于【硬件】还是【软件】
3. 一旦判断足够明确，必须调用 record_issue_type 记录分类并进入下一阶段

如果不明确，可以继续追问，但不要武断下结论。"""


RESOLUTION_SPECIALIST_PROMPT = """你是一名专业的售后支持工程师，正在帮助用户解决设备问题。

【当前阶段：给出解决方案】
已知信息：
- 保修状态 = {warranty_status}
- 问题类型 = {issue_type}

你需要：
1. 如果是【软件问题】：调用 provide_solution 给出清晰的排查/修复步骤（从低风险到高风险）
2. 如果是【硬件问题】：
   - 在保修期：调用 provide_solution 说明官方保修维修流程、备份与注意事项
   - 不在保修期：调用 escalate_to_human 转人工说明付费维修选择

如果用户纠正了信息：
- 用 go_back_to_warranty 回到保修确认
- 用 go_back_to_classification 回到问题分类

要求：回复具体、可执行、条理清晰。"""


STEP_CONFIG = {
    "warranty_collector": {
        "prompt": WARRANTY_COLLECTOR_PROMPT,
        "tools": [record_warranty_status],
        "requires": [],
    },
    "issue_classifier": {
        "prompt": ISSUE_CLASSIFIER_PROMPT,
        "tools": [record_issue_type],
        "requires": ["warranty_status"],
    },
    "resolution_specialist": {
        "prompt": RESOLUTION_SPECIALIST_PROMPT,
        "tools": [provide_solution, escalate_to_human, go_back_to_warranty, go_back_to_classification],
        "requires": ["warranty_status", "issue_type"],
    },
}


# =====================
# 4) Middleware：按 current_step 动态切换配置
# =====================
@wrap_model_call

def apply_step_config(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    current_step = request.state.get("current_step", "warranty_collector")
    stage_config = STEP_CONFIG[current_step]

    for key in stage_config["requires"]:
        if request.state.get(key) is None:
            raise ValueError(f"在进入 {current_step} 之前，必须先设置 {key}")

    system_prompt = stage_config["prompt"].format(**request.state)

    request = request.override(
        system_prompt=system_prompt,
        tools=stage_config["tools"],
    )

    return handler(request)


# =====================
# 5) 创建 Agent（注册所有工具 + checkpointer）
# =====================
all_tools = [
    record_warranty_status,
    record_issue_type,
    provide_solution,
    escalate_to_human,
    go_back_to_warranty,
    go_back_to_classification,
]

agent = create_agent(
    model=model,
    tools=all_tools,
    state_schema=SupportState,
    middleware=[apply_step_config],
    checkpointer=InMemorySaver(),
)


# =====================
# 6) 测试
# =====================
if __name__ == "__main__":
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    print("=== Turn 1: 用户报问题 ===")
    r1 = agent.invoke({"messages": [HumanMessage("你好，我的手机屏幕摔裂了")]}, config)
    for m in r1["messages"]:
        m.pretty_print()

    print("=== Turn 2: 用户回答保修 ===")
    r2 = agent.invoke({"messages": [HumanMessage("还在保修期，去年买的")]}, config)
    for m in r2["messages"]:
        m.pretty_print()

    print("=== Turn 3: 用户描述现象 ===")
    r3 = agent.invoke({"messages": [HumanMessage("裂纹很明显，而且触摸不太灵敏")]}, config)
    for m in r3["messages"]:
        m.pretty_print()

    print("=== Turn 4: 询问怎么处理 ===")
    r4 = agent.invoke({"messages": [HumanMessage("我现在应该怎么处理？")]}, config)
    for m in r4["messages"]:
        m.pretty_print()

=== Turn 1: 用户报问题 ===
================================ Human Message =================================

你好，我的手机屏幕摔裂了
================================== Ai Message ==================================

您好！很遗憾听到您的手机屏幕摔裂了。为了更好地帮助您，我想先确认一下您的手机是否还在保修期内。请问您可以提供一下购买时间或订单信息吗？这样我可以快速帮您查一下保修状态。
=== Turn 2: 用户回答保修 ===
================================ Human Message =================================

你好，我的手机屏幕摔裂了
================================== Ai Message ==================================

您好！很遗憾听到您的手机屏幕摔裂了。为了更好地帮助您，我想先确认一下您的手机是否还在保修期内。请问您可以提供一下购买时间或订单信息吗？这样我可以快速帮您查一下保修状态。
================================ Human Message =================================

还在保修期，去年买的
================================== Ai Message ==================================
Tool Calls:
  record_warranty_status (call_ce0319c1aa4c4bc194d022)
 Call ID: call_ce0319c1aa4c4bc194d022
  Args:
    status: in_warranty
================================= Tool Message =================================
Name: record_warranty_status

保修状态已记录：i